# Fine-Tuning T5-Small for DBpedia Text Classification

This is the original coursework notebook, with execution outputs removed for repository hygiene. The recorded results are preserved in `../results/`.

**Runtime:** Google Colab with a GPU is recommended. **W&B:** Run the login cell only if you want logging to your own account.


In [ ]:
# Install required packages
!pip install --quiet --upgrade transformers huggingface_hub
!pip install -U datasets evaluate wandb
!pip install fsspec==2023.9.2

In [ ]:
# Imports
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer, DataCollatorForSeq2Seq, T5Tokenizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from collections import defaultdict
from datasets import load_dataset, DatasetDict
import pandas as pd
import numpy as np
import random
import wandb
import torch
import os
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
# Authenticate and initialize Weights & Biases
import wandb
wandb.login()
wandb.init(project="LLM-2", name="LLM-2")

In [ ]:
# Tokenize the input and target data
def preprocess(samples):
    inputs = [f"classify: {s['content']}" for s in samples]
    targets = [labels[s["label"]] for s in samples]
    inputs_tokenized = tokenizer(inputs, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
    targets_tokenized = tokenizer(targets, max_length=10, padding="max_length", truncation=True, return_tensors="pt")
    inputs_tokenized["labels"] = targets_tokenized["input_ids"]
    return inputs_tokenized

# Custom dataset class to work with tokenized data
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}
    def __len__(self):
        return len(self.encodings["input_ids"])


# Define evaluation metric using accuracy
accuracy = evaluate.load("accuracy")

# Function to compute evaluation metrics
def compute_metrics(eval_pred):
    logits, labels_ = eval_pred
    preds = np.argmax(logits, axis=-1)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels_, skip_special_tokens=True)
    acc = accuracy.compute(predictions=decoded_preds, references=decoded_labels)
    return {"accuracy": acc["accuracy"]}


In [ ]:
# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load the DBpedia dataset
dataset = load_dataset("dbpedia_14")
labels = dataset["train"].features["label"].names

# Select 1000 samples per class for training
NUM_TRAIN_PER_CLASS = 1000
NUM_TEST = 1000
train_samples = []
label_count = {i: 0 for i in range(len(labels))}

for ex in dataset["train"]:
    lbl = ex["label"]
    if label_count[lbl] < NUM_TRAIN_PER_CLASS:
        train_samples.append(ex)
        label_count[lbl] += 1
    if all(c == NUM_TRAIN_PER_CLASS for c in label_count.values()):
        break

# Select 1000 samples per class for testing
NUM_TEST_PER_CLASS = 1000
test_samples = []
test_label_count = {i: 0 for i in range(len(labels))}

for ex in dataset["test"]:
    lbl = ex["label"]
    if test_label_count[lbl] < NUM_TEST_PER_CLASS:
        test_samples.append(ex)
        test_label_count[lbl] += 1
    if all(c == NUM_TEST_PER_CLASS for c in test_label_count.values()):
        break

# Load tokenizer and model
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)

# Tokenize the training and testing samples
train_enc = preprocess(train_samples)
test_enc = preprocess(test_samples)

# Wrap the tokenized inputs into PyTorch datasets
train_dataset = CustomDataset(train_enc)
test_dataset = CustomDataset(test_enc)

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./t5-dbpedia-output",
    do_eval=True,
    save_strategy="no",
    load_best_model_at_end=False,
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="wandb",
    run_name="t5-finetune-dbpedia"
)


# Train the model using the Trainer API
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()



In [ ]:
label_names = dataset["train"].features["label"].names
label_names_lower = [l.lower() for l in label_names]

# collate_fn و dataloader
def collate_fn(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }

test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)

model.eval()

preds = []
labels_true = []
for batch in tqdm(test_loader, desc="Generating predictions"):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels_tensor = batch["labels"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=10
        )

    decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels_tensor, skip_special_tokens=True)

    decoded_preds = [p.lower().strip() for p in decoded_preds]
    decoded_labels = [l.lower().strip() for l in decoded_labels]

    preds.extend(decoded_preds)
    labels_true.extend(decoded_labels)

#classification_report
report = classification_report(
    labels_true,
    preds,
    labels=label_names_lower,
    target_names=label_names,
    output_dict=True,
    zero_division=0
)
df_report = pd.DataFrame(report).transpose()
print(df_report)

#Confusion Matrix
cm = confusion_matrix(labels_true, preds, labels=label_names_lower)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()

# log to WandB
wandb.log({
    "classification_report": wandb.Table(dataframe=df_report),
    "confusion_matrix": wandb.Image(plt.gcf())
})

plt.show()
wandb.finish()
